<a href="https://colab.research.google.com/github/TejashviDeshmukh/OASIS/blob/main/Task_4_Email_Spam_Detection_With_Machine_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import string
import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import seaborn as sns
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, roc_curve, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from wordcloud import WordCloud

# Download required tokenizers and stopword lists
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

In [ ]:
# Download the dataset directly from a public repository
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"

# Read TSV file directly into pandas
df = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])

# Encode targets: ham -> 0, spam -> 1
df['target'] = df['label'].map({'ham': 0, 'spam': 1})

# Remove duplicate entries
df.drop_duplicates(keep='first', inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Total clean rows: {df.shape[0]}")
print(df['label'].value_counts())

In [ ]:
# Feature extraction for inspection
df['char_count'] = df['message'].apply(len)
df['word_count'] = df['message'].apply(lambda x: len(nltk.word_tokenize(x)))
df['sentence_count'] = df['message'].apply(lambda x: len(nltk.sent_tokenize(x)))

# 1. Summary statistics per class
print("Ham Stats:\n", df[df['target'] == 0][['char_count', 'word_count', 'sentence_count']].describe())
print("\nSpam Stats:\n", df[df['target'] == 1][['char_count', 'word_count', 'sentence_count']].describe())

# 2. Distribution Plot
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.countplot(x='label', data=df, palette='viridis')
plt.title('Spam vs Ham Counts')

plt.subplot(1, 2, 2)
sns.histplot(df[df['target'] == 0]['char_count'], color='blue', label='Ham', kde=True)
sns.histplot(df[df['target'] == 1]['char_count'], color='red', label='Spam', kde=True)
plt.title('Character Count Distribution')
plt.legend()
plt.tight_layout()
plt.show()

# 3. Word Clouds
ham_words = ' '.join(df[df['target'] == 0]['message'])
spam_words = ' '.join(df[df['target'] == 1]['message'])
wc = WordCloud(width=500, height=300, min_font_size=10, background_color='white')

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(wc.generate(ham_words))
plt.axis('off')
plt.title('Most Common Ham Words')

plt.subplot(1, 2, 2)
plt.imshow(wc.generate(spam_words))
plt.axis('off')
plt.title('Most Common Spam Words')
plt.tight_layout()
plt.show()

In [ ]:
ps = PorterStemmer()
stop_words = set(stopwords.words('english'))

def transform_text(text):
    # 1. Lowercase
    text = text.lower()

    # 2. Tokenize
    tokens = nltk.word_tokenize(text)

    # 3. Remove non-alphanumeric, stopwords, and punctuation, then apply Stemming
    cleaned_tokens = [
        ps.stem(word)
        for word in tokens
        if word.isalnum() and word not in stop_words and word not in string.punctuation
    ]

    return " ".join(cleaned_tokens)

# Apply transformation to the entire dataset
df['transformed_message'] = df['message'].apply(transform_text)
df[['message', 'transformed_message']].head()

In [ ]:
# Limit to top 3,000 features to reduce dimensionality
tfidf = TfidfVectorizer(max_features=3000)

X = tfidf.fit_transform(df['transformed_message']).toarray()
y = df['target'].values

# Split into 80% training and 20% testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

In [ ]:
# Drop unnecessary columns from the DataFrame

# columns_to_drop = ["Unnamed: 2", "Unnamed: 3", "Unnamed: 4"]
# df.drop(columns=columns_to_drop, inplace=True)

In [ ]:
df

In [ ]:
# Rename the columns "v1 and "v2" to new names

new_column_names = {"v1":"Category","v2":"Message"}
df.rename(columns = new_column_names,inplace = True)

In [ ]:
df

In [ ]:
# Replace any NaN values in the DataFrame with a space

data = df.where((pd.notnull(df)), ' ')
data.head(10)

In [ ]:
data.describe()

In [ ]:
data.shape

In [ ]:
# Convert the 'label' column values to numerical representation (0 for 'spam' and 1 for 'ham')

data.loc[data['label'] == 'spam', 'label'] = 0
data.loc[data['label'] == 'ham', 'label'] = 1

# Separate the feature (message) and target (label) data

X = data['message']
Y = data['label']

print(X)

In [ ]:
print(Y)

In [ ]:
# Split the data into training and testing sets

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state = 3)
print(X.shape)
print(X_train.shape)
print(X_test.shape)

In [ ]:
# Create a TF-IDF vectorizer to convert text messages into numerical features

feature_extraction = TfidfVectorizer(min_df=1, stop_words="english", lowercase=True)
# Convert the training and testing text messages into numerical features using TF-IDF

X_train_features = feature_extraction.fit_transform(X_train)
X_test_features = feature_extraction.transform(X_test)
# Convert the target values to integers (0 and 1)

Y_train = Y_train.astype("int")
Y_test = Y_test.astype("int")
print(X_train)

In [ ]:
# Initialize and train the Multinomial Naive Bayes model with the new feature set
model = MultinomialNB()
model.fit(X_train_features, Y_train)

# Make predictions on the training data and calculate the accuracy
prediction_on_training_data = model.predict(X_train_features)
accuracy_on_training_data = accuracy_score(Y_train, prediction_on_training_data)
print("Accuracy on training data:",accuracy_on_training_data)

In [ ]:
# Make predictions on the test data and calculate the accuracy

prediction_on_test_data = model.predict(X_test_features)
accuracy_on_test_data = accuracy_score(Y_test,prediction_on_test_data)
print("Accuracy on test data:",accuracy_on_test_data)

In [ ]:
# Test the model with some custom email messages

input_your_mail = ["Congratulations! You have won a free vacation to an exotic destination. Click the link to claim your prize now!"]
input_data_features = feature_extraction.transform(input_your_mail)
prediction = model.predict(input_data_features)
print(prediction)


# Print the prediction result

if (prediction)[0] == 1:
  print("Ham Mail")
else:
  print("Spam Mail")

In [ ]:
input_your_mail = ["Meeting reminder: Tomorrow, 10 AM, conference room. See you there!"]
input_data_features = feature_extraction.transform(input_your_mail)
prediction = model.predict(input_data_features)
print(prediction)


# Print the prediction result

if (prediction)[0] == 1:
  print("Ham Mail")
else:
  print("Spam Mail")

In [ ]:
print(X_train_features)

In [ ]:
# Data visualization - Distribution of Spam and Ham Emails

spam_count = data[data['label'] == 0].shape[0]
ham_count = data[data['label'] == 1].shape[0]

plt.bar(['Spam', 'Ham'], [spam_count, ham_count])
plt.xlabel('Email Type')
plt.ylabel('Count')
plt.title('Distribution of Spam and Ham Emails')
plt.show()

In [ ]:
probabilities = model.predict_proba(X_test_features)[:, 1]
fpr, tpr, thresholds = roc_curve(Y_test, probabilities)
roc_auc = roc_auc_score(Y_test, probabilities)

plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend()
plt.show()

In [ ]:
# Initialize and train
model = MultinomialNB()
model.fit(X_train, y_train)

# Evaluate predictions
y_pred = model.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Plot Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(4, 3))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
from collections import Counter

stop_words = set(stopwords.words('english'))
spam_words = " ".join(data[data['label'] == 0]['message']).split()
ham_words = " ".join(data[data['label'] == 1]['message']).split()

spam_word_freq = Counter([word.lower() for word in spam_words if word.lower() not in stop_words and word.isalpha()])

plt.figure(figsize=(10, 6))
plt.bar(*zip(*spam_word_freq.most_common(10)), color='purple')
plt.xlabel('Words')
plt.ylabel('Frequency')
plt.title('Top 10 Most Common Words in Spam Emails')
plt.xticks(rotation=45)
plt.show()

In [ ]:
ham_word_freq = Counter([word.lower() for word in ham_words if word.lower() not in stop_words and word.isalpha()])

plt.figure(figsize=(10, 6))
plt.bar(*zip(*ham_word_freq.most_common(10)), color='orange')
plt.xlabel('Words')
plt.ylabel('Frequency')
plt.title('Top 10 Most Common Words in Ham Emails')
plt.xticks(rotation=45)
plt.show()
